In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import classification_report

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
try:
    import sktime
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "sktime", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import data_utils
from baselines_utils import (
    evaluate_holdout
)
from base_utils_qwen import (
    SequenceExtractor,
    competition_scorer as bfrb_competition_scorer,
    evaluate_holdout
)
from ensemble_utils import HierarchicalBFRBEnsemble
from single_minirocket import SingleMiniRocketClassifier
from sklearn.metrics import f1_score, make_scorer

E0000 00:00:1781480570.529500      30 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781480570.599226      30 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781480571.171128      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781480571.171171      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781480571.171174      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781480571.171176      30 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
TRAIN_SINGLE = True   # Set to True to train/evaluate the Single MiniRocket
TRAIN_ENSEMBLE = False # Set to True to train/evaluate the Hierarchical Ensemble

preprocess_handness = False

# Options: "bfrb", "orientation", "gesture", "gesture_action", "gesture_position", "is_target", "phase"
# Note: Ensure the chosen column exists in your train_df before running!
TARGET_COL = "bfrb"  
orientation_col = "orientation"

search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 3
n_iter = 10
train_size = 0.2
error_score_constant = 0.0
verbose = 3
do_cross_val = False

full_wham = True
if full_wham:
    test_size = 1 - train_size
else:
    test_size = min(0.25, 1 - train_size)

cv_object = GroupKFold(n_splits=n_splits) if do_cross_val else GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# DYNAMIC SCORER SELECTION
if TARGET_COL == 'bfrb':
    scorer = bfrb_competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

print(f"TRAIN_SINGLE: {TRAIN_SINGLE} | TRAIN_ENSEMBLE: {TRAIN_ENSEMBLE}")
print(f"Target Column: {TARGET_COL}")
print(f"Search mode: {search_mode}")

TRAIN_SINGLE: True | TRAIN_ENSEMBLE: False
Target Column: bfrb
Search mode: grid


In [3]:
data_root = data_utils.find_data_root()
raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if preprocess_handness: 
# Handedness & Upside-down corrections
    if "handedness" in train_demo_df.columns:
        train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
        left_handed_mask = train_df["handedness"].eq(0)
        train_df.loc[left_handed_mask, "acc_x"] *= -1.0
        train_df = train_df.drop(columns=["handedness"])

    upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
    train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
    train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

# Create alternative target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]


Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [4]:
# Get unique sequences for splitting to prevent leakage
sequences = train_df[['sequence_id', 'is_target', TARGET_COL, orientation_col]].drop_duplicates()
seq_ids = sequences['sequence_id'].unique()

if not do_cross_val:
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(seq_ids, groups=seq_ids))
    train_seqs = seq_ids[train_idx]
    test_seqs = seq_ids[test_idx]
else:
    train_seqs = seq_ids # For CV, we use all data in the search object
    test_seqs = []

train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy() if len(test_seqs) > 0 else pd.DataFrame()

X_train = train_sample_df
X_test = hold_out_df

# Setup y based on MODE
if TRAIN_ENSEMBLE:
    y_train = train_sample_df[["sequence_id", "is_target", orientation_col, TARGET_COL]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", orientation_col, TARGET_COL]].copy() if len(hold_out_df) > 0 else pd.DataFrame()
else:
    y_train = train_sample_df[["sequence_id", "is_target", TARGET_COL]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", TARGET_COL]].copy() if len(hold_out_df) > 0 else pd.DataFrame()

groups = X_train["sequence_id"]

print(f"Train sequences: {X_train['sequence_id'].nunique()} | Test sequences: {X_test['sequence_id'].nunique() if len(X_test) > 0 else 0}")

Train sequences: 1630 | Test sequences: 6521


In [5]:
# ============================================================
# PARAMETER SPACE DEFINITION
# ============================================================
if TRAIN_ENSEMBLE:
    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        bayes_extractor_space = {
            "extractor__acc_modes": Categorical(["smoothed|velocity|jerk"]),
            "extractor__rotation_modes": Categorical([ "quaternion|angular_velocity|euler"]),
            "extractor__tof_modes": Categorical(["pooled_stats|sensor_stats"]),
            "extractor__thm_modes": Categorical(["centered_diff"]),
            "extractor__sampling_rate": Categorical([200]),
            "extractor__maxlen": Categorical([180]),
            "extractor__window_size": Categorical([50]),
            "extractor__clip_value": Categorical([150.0]),
            "extractor__interp_mode": Categorical(["linear"]),
            "extractor__motion_filter_mode": Categorical([ "kalman"]),
            "extractor__use_dead_reckoning": Categorical([True]),
            "extractor__dead_reckoning_detrend": Categorical([True]),
            "extractor__kalman_process_noise": Real(1e-6, 1e-1, prior="log-uniform"),
            "extractor__kalman_measurement_noise": Real(1e-3, 1e2, prior="log-uniform"),
        }

        ensemble_param_space = {
            **bayes_extractor_space,
            # Layer 1: Binary BFRB
            "classifier__l1_num_kernels": Categorical([3700]),
            # "classifier__l1_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l1_class_weight": Categorical(["balanced", None]),
            # "classifier__l1_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            # Layer 2: Orientation
            "classifier__l2_num_kernels": Categorical([3700]),
            # "classifier__l2_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l2_class_weight": Categorical(["balanced", None]),
            # "classifier__l2_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            # Layer 3: BFRB per Orientation (Models 1-4)
            "classifier__l3_1_num_kernels": Integer(1000, 3000),
            # "classifier__l3_1_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_1_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            "classifier__l3_2_num_kernels": Integer(1000, 3000),
            # "classifier__l3_2_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_2_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            "classifier__l3_3_num_kernels": Integer(1000, 3000),
            # "classifier__l3_3_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_3_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            "classifier__l3_4_num_kernels": Integer(1000, 3000),
            # "classifier__l3_4_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_4_feature_selection_percentile": Categorical([None, 25, 50, 75]), # ADDED
            
            # "classifier__l3_class_weight": Categorical(["balanced", None]),
        }

    else:  # GRID SEARCH
        grid_extractor_space = {
            "extractor__acc_modes": ["smoothed|velocity|displacement|jerk"],
            "extractor__rotation_modes": ["quaternion|angular_velocity|euler"], # FIXED from "raw"
            "extractor__tof_modes": ["pooled_stats|sensor_stats"],                     # FIXED from "raw"
            "extractor__thm_modes": ["centered_diff|diff"],                              # FIXED typo from "raww"
            "extractor__sampling_rate": [200],
            "extractor__maxlen": [160],
            "extractor__window_size": [40],
            "extractor__clip_value": [None],
            "extractor__interp_mode": ["linear"],
            "extractor__motion_filter_mode": [None],
            "extractor__use_dead_reckoning": [False],
            "extractor__dead_reckoning_detrend": [False],
            "extractor__kalman_process_noise": [1e-3],
            "extractor__kalman_measurement_noise": [1e-1],
        }

        ensemble_param_space = {
            **grid_extractor_space,
            # Layer 1
            "classifier__l1_num_kernels": [2000],
            "classifier__l1_alpha": [1e3],
            "classifier__l1_class_weight": ["balanced"],
            "classifier__l1_feature_selection_percentile": [50], # ADDED
            
            # Layer 2
            "classifier__l2_num_kernels": [2000],
            "classifier__l2_alpha": [1e4],
            "classifier__l2_class_weight": ["balanced"],
            "classifier__l2_feature_selection_percentile": [50], # ADDED
            
            # Layer 3
            "classifier__l3_1_num_kernels": [2000],
            "classifier__l3_1_alpha": [1e4],
            "classifier__l3_1_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_2_num_kernels": [2000],
            "classifier__l3_2_alpha": [1e4],
            "classifier__l3_2_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_3_num_kernels": [2000],
            "classifier__l3_3_alpha": [1e4],
            "classifier__l3_3_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_4_num_kernels": [2000],
            "classifier__l3_4_alpha": [1e4],
            "classifier__l3_4_feature_selection_percentile": [50], # ADDED
            
            "classifier__l3_class_weight": ["balanced"],
        }

# ... (Keep your existing bayes_extractor_space / grid_extractor_space definitions) ...

if TRAIN_SINGLE:
# ============================================================
# SINGLE MINI-ROCKET PARAMETER SPACE
# ============================================================
    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        # Base Extractor Space (Bayesian)
        bayes_extractor_space = {
            "extractor__acc_modes": Categorical(["raw", "smoothed|velocity|jerk"]),
            "extractor__rotation_modes": Categorical(["quaternion|angular_velocity|euler", "raw"]),
            "extractor__tof_modes": Categorical(["pooled_stats|sensor_stats"]),
            "extractor__thm_modes": Categorical(["centered_diff"]),
            "extractor__sampling_rate": Integer(20, 200),
            "extractor__maxlen": Integer(100, 200),
            "extractor__window_size": Integer(10, 50),
            "extractor__clip_value": Real(30.0, 100.0, prior="linear"),
            "extractor__interp_mode": Categorical(["linear"]),
            "extractor__motion_filter_mode": Categorical(["kalman", None]),
            "extractor__use_dead_reckoning": Categorical([False]),
            "extractor__dead_reckoning_detrend": Categorical([False]),
            "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
            "extractor__kalman_measurement_noise": Real(1e-2, 1e-1, prior="log-uniform"),
            "extractor__padding_value": Categorical([0.0]), # CRITICAL for MiniRocket
        }

        # Merge with Classifier Space
        single_param_space = {
            **bayes_extractor_space,
            "classifier__target_col": Categorical([TARGET_COL]),
            "classifier__num_kernels": Integer(1000, 2000),
            "classifier__alpha": Real(1e1, 1e4, prior="log-uniform"),
            "classifier__feature_selection_percentile": Categorical([None, 25, 50, 75]),
            "classifier__class_weight": Categorical(["balanced", None]),
        }

    else:  # GRID SEARCH
        # Base Extractor Space (Grid)
        grid_extractor_space = {
            # "extractor__acc_modes": ["smoothed|velocity|displacement|jerk"],
            # "extractor__rotation_modes": ["quaternion|angular_velocity|euler"],
            # "extractor__tof_modes": ["pooled_stats|sensor_stats"],
            # "extractor__thm_modes": ["centered_diff|diff"],
            "extractor__acc_modes": ["raw"],
            "extractor__rotation_modes": ["raw"],
            "extractor__tof_modes": ["raw"],
            "extractor__thm_modes": ["raw"],
            "extractor__sampling_rate": [20],
            "extractor__maxlen": [180],
            "extractor__window_size": [20],
            "extractor__clip_value": [None],
            "extractor__interp_mode": ["linear"],
            "extractor__motion_filter_mode": [None],
            "extractor__use_dead_reckoning": [False],
            "extractor__dead_reckoning_detrend": [False],
            "extractor__kalman_process_noise": [1e-3],
            "extractor__kalman_measurement_noise": [1e-1],
            "extractor__padding_value": [0.0], # CRITICAL for MiniRocket
        }

        # Merge with Classifier Space
        single_param_space = {
            **grid_extractor_space,
            "classifier__target_col": [TARGET_COL],
            "classifier__num_kernels": [84*1],
            "classifier__alpha": [1e1, 1e4],
            "classifier__feature_selection_percentile": [None, 50, 20],
            "classifier__class_weight": ["balanced"],
        }

In [6]:
# ============================================================
# MODEL TRAINING & EVALUATION LOOP
# ============================================================
results_list = []
fitted_models = {}

# -----------------------------------------------------------------
# 1. SINGLE MINI-ROCKET MODEL
# -----------------------------------------------------------------
if TRAIN_SINGLE:
    print(f"\n--- Training: Single MiniRocket (Target: {TARGET_COL}) ---")
    
    single_pipe = Pipeline([
        ("extractor", SequenceExtractor(
            acc_modes="smoothed|velocity|jerk",
        )),
        ("classifier", SingleMiniRocketClassifier(
            target_col=TARGET_COL,
            num_kernels=2000,
            alpha=1e3,
            feature_selection_percentile=50,
            class_weight="balanced",
            random_state=random_state
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        single_search = BayesSearchCV(
            single_pipe, single_param_space, n_iter=n_iter, scoring=scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        single_search = GridSearchCV(
            single_pipe, single_param_space, scoring=scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    single_search.fit(X_train, y_train, groups=groups)
    fitted_models["Single MiniRocket"] = single_search.best_estimator_
    
    y_pred_single = single_search.predict(X_test)
    
    # Dynamic Evaluation based on TARGET_COL
    if TARGET_COL == 'bfrb':
        eval_dict_single = evaluate_holdout(y_test, y_pred_single, target_col=TARGET_COL, verbose=True)
        score_single = eval_dict_single.get("competition_score", 0)
    else:
        from sklearn.metrics import f1_score, classification_report
        y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
        print("\nClassification Report:")
        print(classification_report(y_test_seq[TARGET_COL], y_pred_single, zero_division=0))
        score_single = f1_score(y_test_seq[TARGET_COL], y_pred_single, average="macro", zero_division=0)
        print(f"Macro F1 Score: {score_single:.4f}")
    
    print(f"Single MiniRocket Best CV Score: {single_search.best_score_:.4f} | Holdout Score: {score_single:.4f}")
    print(f"Best Params: {single_search.best_params_}")
    
    results_list.append({
        "Model": "Single MiniRocket",
        "Target": TARGET_COL,
        "CV Score": single_search.best_score_,
        "Holdout Score": score_single,
        "Best Params": single_search.best_params_,
    })

# -----------------------------------------------------------------
# 2. HIERARCHICAL ENSEMBLE MODEL (Strictly for BFRB)
# -----------------------------------------------------------------
if TRAIN_ENSEMBLE:
    print("\n--- Training: Hierarchical BFRB Ensemble ---")
    
    l3_override_dict = None 

    ensemble_pipe = Pipeline([
        ("extractor", SequenceTensorExtractor(
            acc_modes="smoothed|velocity|jerk",
            rotation_modes="quaternion|angular_velocity",
            tof_modes="pooled_stats",
            thm_modes="centered_diff",
            sampling_rate=50,
            maxlen=150,
            window_size=40,
            clip_value=50.0,
            interp_mode="linear",
            motion_filter_mode="kalman",
            use_dead_reckoning=False,
            dead_reckoning_detrend=False,
            kalman_process_noise=1e-3,
            kalman_measurement_noise=1e-1
        )),
        ("classifier", HierarchicalBFRBEnsemble(
            orientation_col=orientation_col,
            target_col="bfrb", # Ensemble is strictly designed for BFRB
            l3_params_dict=l3_override_dict,
            random_state=random_state
        ))
    ])

    if search_mode == "bayesian" and SKOPT_AVAILABLE:
        ensemble_search = BayesSearchCV(
            ensemble_pipe, ensemble_param_space, n_iter=n_iter, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, random_state=random_state,
            error_score=error_score_constant, verbose=verbose, return_train_score=True
        )
    else:
        ensemble_search = GridSearchCV(
            ensemble_pipe, ensemble_param_space, scoring=competition_scorer,
            cv=cv_object, n_jobs=1, error_score=error_score_constant, verbose=verbose, return_train_score=True
        )

    ensemble_search.fit(X_train, y_train, groups=groups)
    fitted_models["Hierarchical Ensemble"] = ensemble_search.best_estimator_
    
    y_pred_ens = ensemble_search.predict(X_test)
    
    eval_dict_ens = evaluate_holdout(y_test, y_pred_ens, target_col="bfrb", verbose=True)
    score_ens = eval_dict_ens.get("competition_score", eval_dict_ens.get("holdout_score", 0))
    
    print(f"Hierarchical Ensemble Best CV Score: {ensemble_search.best_score_:.4f} | Holdout Score: {score_ens:.4f}")
    print(f"Best Params: {ensemble_search.best_params_}")
    
    results_list.append({
        "Model": "Hierarchical Ensemble",
        "Target": "bfrb",
        "CV Score": ensemble_search.best_score_,
        "Holdout Score": score_ens,
        "Best Params": ensemble_search.best_params_,
    })

# ============================================================
# FINAL SUMMARY
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "="*60)
print("BASELINES SUMMARY")
print("="*60)
print(results_df.to_string(index=False))


--- Training: Single MiniRocket (Target: bfrb) ---
Fitting 1 folds for each of 6 candidates, totalling 6 fits
[CV 1/1] END classifier__alpha=10.0, classifier__class_weight=balanced, classifier__feature_selection_percentile=None, classifier__num_kernels=84, classifier__target_col=bfrb, extractor__acc_modes=raw, extractor__clip_value=None, extractor__dead_reckoning_detrend=False, extractor__interp_mode=linear, extractor__kalman_measurement_noise=0.1, extractor__kalman_process_noise=0.001, extractor__maxlen=180, extractor__motion_filter_mode=None, extractor__padding_value=0.0, extractor__rotation_modes=raw, extractor__sampling_rate=20, extractor__thm_modes=raw, extractor__tof_modes=raw, extractor__use_dead_reckoning=False, extractor__window_size=20;, score=(train=0.989, test=0.495) total time=  55.3s
[CV 1/1] END classifier__alpha=10.0, classifier__class_weight=balanced, classifier__feature_selection_percentile=50, classifier__num_kernels=84, classifier__target_col=bfrb, extractor__acc_m

In [7]:
# ============================================================
# FINAL SUMMARY
# ============================================================
results_df = pd.DataFrame(results_list).sort_values("Holdout Score", ascending=False, na_position="last")
results_df.to_csv(results_dir / f"baselines_summary_{timestamp}.csv", index=False)

print("\n" + "="*60)
print("BASELINES SUMMARY")
print("="*60)
print(results_df.to_string(index=False))

if len(results_df) > 0:
    best_name = results_df.iloc[0]["Model"]
    print(f"\nDetailed holdout eval for best model: {best_name}")
    best_model = fitted_models[best_name]
    
    y_pred = best_model.predict(X_test)
    
    # Collapse y_test to sequence level to match the length of y_pred
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
    
    print("\nClassification Report:")
    print(classification_report(y_test_seq[TARGET_COL], y_pred, zero_division=0))


BASELINES SUMMARY
            Model Target  CV Score  Holdout Score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        Best Params
Single MiniRocket   bfrb  0.551149       0.631719 {'classifier__alpha': 10.0, 'classifier__class_weight': 'balanced', 'classifier__feature_selection_percentile': 20, 'classifier__num_kernels': 84, 'classifier__target_col': '